# 0.1 — Proteins as Tensors

**Mechanism of the day:** how a protein becomes numbers a model can generate.

Before we can *generate* proteins, we need to know what the model is emitting.
There are two views, and every method in this gym commits to one of them:

- **Sequence view** — a protein is a string over a 20-letter alphabet (the amino
  acids). Becomes an integer array `[L]`, i.e. *discrete* data. (Autoregressive
  models, masked LMs, discrete diffusion live here.)
- **Structure view** — a protein is a set of 3D atom coordinates `[L, 3]`, i.e.
  *continuous* data living in 3D space. (Structure diffusion lives here.)

This notebook builds both, plus the two objects you will see over and over:
the **distance/contact map** and the per-residue **SE(3) frame**.

**How to use this notebook:** read the concept, do the reps (functions that
`raise NotImplementedError`), make the checkpoints (`assert`s) pass. Solutions
are at the very bottom — don't peek until you've tried.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (5, 4)

## Part 1 — Sequence as a tensor

There are 20 standard amino acids, each with a one-letter code. A model over
sequences works with *indices* (0..19), and often with *one-hot* vectors.

In [ ]:
# The 20 standard amino acids, one-letter codes.
AA = 'ACDEFGHIKLMNPQRSTVWY'
STOI = {a: i for i, a in enumerate(AA)}   # string -> int
ITOS = {i: a for i, a in enumerate(AA)}   # int -> string
VOCAB = len(AA)                            # 20

# A toy 'protein' to practice on.
seq = 'MKTAYIAKQR'
print(seq, '| length', len(seq), '| vocab size', VOCAB)

### Rep 1 — `seq_to_indices`
Map an amino-acid string to a 1-D integer array. Example: `'AC' -> [0, 1]`.

In [ ]:
def seq_to_indices(s):
    '''Map an amino-acid string to a 1-D int array of indices in [0, 20).'''
    # YOUR CODE HERE (one line is enough)
    raise NotImplementedError

# --- checkpoint ---
idx = seq_to_indices(seq)
assert idx.shape == (len(seq),), 'wrong shape'
assert idx.dtype.kind in 'iu', 'should be integers'
assert idx.tolist() == [STOI[c] for c in seq]
print('idx =', idx, '✓')

### Rep 2 — `one_hot`
Turn indices `[L]` into a float one-hot matrix `[L, 20]`. Each row sums to 1.
This is the *distribution* an untrained model would have to learn to place mass on.

In [ ]:
def one_hot(idx, vocab=VOCAB):
    '''Turn a 1-D int array [L] into a float one-hot matrix [L, vocab].'''
    # YOUR CODE HERE
    raise NotImplementedError

x = one_hot(idx)
assert x.shape == (len(seq), VOCAB)
assert np.allclose(x.sum(axis=1), 1.0)
assert x[0, idx[0]] == 1.0
print('one-hot shape', x.shape, '✓')

## Part 2 — Structure as a tensor

Now the continuous view. A protein backbone is a chain of atoms in 3D. We'll use
just the alpha-carbon (CA) of each residue: coordinates `[L, 3]`.

Instead of downloading a PDB file, we synthesize an **idealized alpha-helix** —
one of the two fundamental shapes proteins fold into. It is literally a helix:
each residue steps ~1.5 A along an axis and rotates ~100 degrees around it. Real
helices look almost exactly like this, which is why the maps below are so clean.

In [ ]:
def make_alpha_helix(L, rise=1.5, turn_deg=100.0, radius=2.3):
    '''Return CA coordinates [L, 3] of an idealized alpha helix.'''
    t = np.arange(L)
    ang = np.deg2rad(turn_deg) * t
    x = radius * np.cos(ang)
    y = radius * np.sin(ang)
    z = rise * t
    return np.stack([x, y, z], axis=1)

ca = make_alpha_helix(30)
print('CA coords shape', ca.shape)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot(ca[:, 0], ca[:, 1], ca[:, 2], '-o', ms=3)
ax.set_title('Idealized alpha-helix backbone (CA atoms)')
plt.show()

### Rep 3 — `distance_matrix`
The **distance map** `D[i,j] = ||x_i - x_j||` is how models 'see' structure
without caring about absolute position or orientation (it is rotation/translation
invariant). Use broadcasting: `coords[:, None, :]` has shape `[L, 1, 3]`.

In [ ]:
def distance_matrix(coords):
    '''Pairwise Euclidean distances. coords [L,3] -> D [L,L].'''
    # YOUR CODE HERE
    raise NotImplementedError

D = distance_matrix(ca)
assert D.shape == (len(ca), len(ca))
assert np.allclose(np.diag(D), 0.0)
assert np.allclose(D, D.T)
plt.imshow(D, cmap='viridis'); plt.colorbar(label='angstroms')
plt.title('Distance map'); plt.xlabel('residue j'); plt.ylabel('residue i'); plt.show()
print('✓ notice the diagonal band: neighbors in sequence are close in space')

### Rep 4 — `contact_map`
A **contact map** is the thresholded distance map: `True` where two residues are
within ~8 A (and not the same residue). Many structure models predict/consume
this binary object. Build it on top of `distance_matrix`.

In [ ]:
def contact_map(coords, threshold=8.0):
    '''Boolean [L,L]: True where residues i != j are within threshold angstroms.'''
    # YOUR CODE HERE
    raise NotImplementedError

C = contact_map(ca)
assert C.dtype == bool
assert not C.diagonal().any(), 'a residue should not contact itself'
plt.imshow(C, cmap='gray_r'); plt.title('Contact map (< 8 A)')
plt.xlabel('residue j'); plt.ylabel('residue i'); plt.show()

## Part 3 (stretch) — the per-residue SE(3) frame

Here's the object that structure-diffusion models (RFdiffusion, FrameFlow,
FoldFlow) actually generate: not raw coordinates, but a **rigid frame** per
residue — a rotation `R` (3x3) plus a translation `t` (3). It says 'this residue
sits *here* and is oriented *this way*'.

We build it from three backbone atoms (N, CA, C) with Gram-Schmidt:
- `t` = the CA position
- `e1` = unit vector along `(C - CA)`
- `e2` = part of `(N - CA)` orthogonal to `e1`, normalized
- `e3` = `e1 x e2`

`R = [e1 e2 e3]` (as columns) is then a valid rotation. Generating on the SE(3)
manifold instead of raw xyz is *the* idea that made structure diffusion work.

In [ ]:
def rigid_from_3points(N, CA, C):
    '''Build a residue frame (R [3,3], t [3]) from backbone atoms via Gram-Schmidt.'''
    # YOUR CODE HERE
    raise NotImplementedError

# random non-degenerate backbone triplet to test the geometry
N  = rng.normal(size=3)
CA = rng.normal(size=3)
C  = rng.normal(size=3)
R, t = rigid_from_3points(N, CA, C)
assert np.allclose(R @ R.T, np.eye(3), atol=1e-6), 'R must be orthonormal'
assert np.isclose(np.linalg.det(R), 1.0, atol=1e-6), 'R must be a proper rotation (det +1)'
assert np.allclose(t, CA)
print('valid SE(3) frame ✓')

## Reflection — what just transferred

- **Discrete vs continuous data** is the fork that decides which generative
  machinery you use. You built both representations by hand.
- The **distance map** is an *invariant* representation — the same trick shows up
  anywhere geometry matters (point clouds, molecules, robotics).
- The **SE(3) frame** is why 'generate a protein structure' became tractable:
  you generate on a manifold of rigid motions, not in raw coordinate soup. Keep
  this object in mind — rep 1.6 generates exactly these.

**Next rep:** `0.2 The oracle mindset` — formalize what 'controllable
generation' even means, so Track 2 has a target to aim at.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def seq_to_indices(s):
    return np.array([STOI[c] for c in s])

def one_hot(idx, vocab=VOCAB):
    return np.eye(vocab)[idx]

def distance_matrix(coords):
    diff = coords[:, None, :] - coords[None, :, :]   # [L, L, 3]
    return np.linalg.norm(diff, axis=-1)             # [L, L]

def contact_map(coords, threshold=8.0):
    C = distance_matrix(coords) < threshold
    np.fill_diagonal(C, False)
    return C

def rigid_from_3points(N, CA, C):
    e1 = C - CA; e1 = e1 / np.linalg.norm(e1)
    v  = N - CA
    e2 = v - (e1 @ v) * e1; e2 = e2 / np.linalg.norm(e2)
    e3 = np.cross(e1, e2)
    R = np.stack([e1, e2, e3], axis=1)   # columns are the frame axes
    return R, CA

print('reference solutions loaded — re-run the checkpoint cells above')